## Importing Libraries

In [23]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
%matplotlib inline 

## Setting Paths

In [24]:
DATAPATH = "../data/raw/"
OUTPUTPATH = "../data/processed/"
os.makedirs(OUTPUTPATH, exist_ok=True)

DATAFILE_NAME = os.path.join(DATAPATH, "medical-appointments-no-show-en.csv")
ICD10FILE_NAME = os.path.join(DATAPATH, "icd10_2019.csv")

## 1. Loading Data

In [25]:
if os.path.exists(ICD10FILE_NAME):
    icd10 = pd.read_csv(ICD10FILE_NAME)
    icd10 = icd10[["sub-code", "definition"]]

if os.path.exists(DATAFILE_NAME):
    data = pd.read_csv(DATAFILE_NAME)
    display(data.head())

,specialty,appointment_time,gender,appointment_date,no_show,no_show_reason,disability,date_of_birth,entry_service_date,city,...,over_60_years_old,patient_needs_companion,average_temp_day,average_rain_day,max_temp_day,max_rain_day,rainy_day_before,storm_day_before,rain_intensity,heat_intensity
0,physiotherapy,13:20,M,09/09/2021,yes,surto,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
1,psychotherapy,13:20,M,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
2,speech therapy,13:20,F,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
3,physiotherapy,13:20,F,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
4,physiotherapy,14:00,M,09/09/2021,no,NaN,motor,10/10/1954,5/2/2020,B. CAMBORIU,...,1,1,20.75,0.01,23.7,0.2,1,1,no_rain,mild


## 2. Removing Duplicates
> This dataset cotains **2.921 linhas duplicadas** found in the data quality diagnosis. Removing before aany transformation to prevent distortions in the future analysis.

In [26]:
before = len(data)
data = data.drop_duplicates()
after = len(data)
print(f"Lines Removed : {before - after} | Remaining Lines: {after}")

Lines Removed : 2921 | Remaining Lines: 46672


## 3. Standardizing Types and Values

- Datas to `YYYY-MM-DD` format
- `no_show`: `no -> 0`, `yes -> 1`
- `age`: removing unecessary decimals
- `no_show_reason`: strings correction

In [27]:
if os.path.exists(DATAFILE_NAME):
    data = pd.read_csv(DATAFILE_NAME)
    display(data)

,specialty,appointment_time,gender,appointment_date,no_show,no_show_reason,disability,date_of_birth,entry_service_date,city,...,over_60_years_old,patient_needs_companion,average_temp_day,average_rain_day,max_temp_day,max_rain_day,rainy_day_before,storm_day_before,rain_intensity,heat_intensity
0,physiotherapy,13:20,M,09/09/2021,yes,surto,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
1,psychotherapy,13:20,M,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
2,speech therapy,13:20,F,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
3,physiotherapy,13:20,F,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
4,physiotherapy,14:00,M,09/09/2021,no,NaN,motor,10/10/1954,5/2/2020,B. CAMBORIU,...,1,1,20.75,0.01,23.7,0.2,1,1,no_rain,mild
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49588,NaN,10:20,M,01/02/2022,yes,pai tem médico,intellectual,9/3/2009,5/11/2019,ITAJAÍ,...,0,0,NaN,NaN,NaN,NaN,0,0,no_rain,heavy_cold
49589,NaN,10:20,M,01/02/2022,yes,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,0,0,no_rain,heavy_cold
49590,NaN,11:00,M,01/02/2022,no,NaN,intellectual,7/3/2016,11/2/2020,BOMBINHAS,...,0,1,NaN,NaN,NaN,NaN,0,0,no_rain,heavy_cold
49591,psychotherapy,11:00,M,01/02/2022,yes,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,0,0,no_rain,heavy_cold


In [28]:
if not data.empty:

    # Data
    data["appointment_date"] = pd.to_datetime(
        data["appointment_date"], errors="coerce", format="%d/%m/%Y"
    ).dt.strftime("%Y-%m-%d")

    data["entry_service_date"] = pd.to_datetime(
        data["entry_service_date"], errors="coerce", format="%d/%m/%Y"
    ).dt.strftime("%Y-%m-%d")

    # Target variable
    data["no_show"] = data["no_show"].map({"no": 0, "yes": 1})

    # Ages
    data["age"] = (
        data["age"]
        .apply(
            lambda row: (
                str(row).split(".")[0]
                if isinstance(row, (int, float)) and not pd.isna(row)
                else row
            )
        )
        .replace("nan", np.nan)
    )

    data.loc[
        (data["no_show_reason"].str.contains("yes", na=False)) & (data["no_show"] == 1),
        "no_show_reason",
    ] = data["no_show_reason"].str.replace("yes", "nao", regex=False)

    display(data.head())

,specialty,appointment_time,gender,appointment_date,no_show,no_show_reason,disability,date_of_birth,entry_service_date,city,...,over_60_years_old,patient_needs_companion,average_temp_day,average_rain_day,max_temp_day,max_rain_day,rainy_day_before,storm_day_before,rain_intensity,heat_intensity
0,physiotherapy,13:20,M,2021-09-09,1,surto,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
1,psychotherapy,13:20,M,2021-09-09,0,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
2,speech therapy,13:20,F,2021-09-09,0,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
3,physiotherapy,13:20,F,2021-09-09,0,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
4,physiotherapy,14:00,M,2021-09-09,0,NaN,motor,10/10/1954,2020-02-05,B. CAMBORIU,...,1,1,20.75,0.01,23.7,0.2,1,1,no_rain,mild


In [29]:
if not data.empty:

    data.drop(
        columns=[
            "average_rain_day",
            "average_temp_day",
            "max_rain_day",
            "date_of_birth",
            "sub-code",
            "storm_day_before",
        ],
        errors="ignore",
        inplace=True,
    )

    data.loc[
        data["no_show_reason"].str.contains("yes") & (data["no_show"] == 1),
        "no_show_reason",
    ] = data["no_show_reason"].str.replace("yes", "não", regex=False)
    display(data)

,specialty,appointment_time,gender,appointment_date,no_show,no_show_reason,disability,entry_service_date,city,icd,...,appointment_year,appointment_shift,age,under_12_years_old,over_60_years_old,patient_needs_companion,max_temp_day,rainy_day_before,rain_intensity,heat_intensity
0,physiotherapy,13:20,M,2021-09-09,1,surto,NaN,NaN,NaN,NaN,...,2021,afternoon,NaN,0,0,0,23.7,1,no_rain,mild
1,psychotherapy,13:20,M,2021-09-09,0,NaN,NaN,NaN,NaN,NaN,...,2021,afternoon,NaN,0,0,0,23.7,1,no_rain,mild
2,speech therapy,13:20,F,2021-09-09,0,NaN,NaN,NaN,NaN,NaN,...,2021,afternoon,NaN,0,0,0,23.7,1,no_rain,mild
3,physiotherapy,13:20,F,2021-09-09,0,NaN,NaN,NaN,NaN,NaN,...,2021,afternoon,NaN,0,0,0,23.7,1,no_rain,mild
4,physiotherapy,14:00,M,2021-09-09,0,NaN,motor,2020-02-05,B. CAMBORIU,I67,...,2021,afternoon,68,0,1,1,23.7,1,no_rain,mild
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49588,NaN,10:20,M,2022-02-01,1,pai tem médico,intellectual,2019-11-05,ITAJAÍ,F84.5,...,2022,morning,13,0,0,0,NaN,0,no_rain,heavy_cold
49589,NaN,10:20,M,2022-02-01,1,NaN,NaN,NaN,NaN,NaN,...,2022,morning,NaN,0,0,0,NaN,0,no_rain,heavy_cold
49590,NaN,11:00,M,2022-02-01,0,NaN,intellectual,2020-02-11,BOMBINHAS,F84,...,2022,morning,6,1,0,1,NaN,0,no_rain,heavy_cold
49591,psychotherapy,11:00,M,2022-02-01,1,NaN,NaN,NaN,NaN,NaN,...,2022,morning,NaN,0,0,0,NaN,0,no_rain,heavy_cold


## 4. Enriching with ICD-10
> Merge to bring a textual description to ICDs. Maintaining previous data with `left join` to not lose any data without ICD

In [30]:
if not data.empty and not icd10.empty:
    data = pd.merge(
        data,
        icd10.rename(columns={"definition": "icd_description"}),
        left_on="icd",
        right_on="sub-code",
        how="left",
    )
    data.drop(columns=["sub-code"], errors="ignore", inplace=True)
    display(data.head())

,specialty,appointment_time,gender,appointment_date,no_show,no_show_reason,disability,entry_service_date,city,icd,...,appointment_shift,age,under_12_years_old,over_60_years_old,patient_needs_companion,max_temp_day,rainy_day_before,rain_intensity,heat_intensity,icd_description
0,physiotherapy,13:20,M,2021-09-09,1,surto,NaN,NaN,NaN,NaN,...,afternoon,NaN,0,0,0,23.7,1,no_rain,mild,NaN
1,psychotherapy,13:20,M,2021-09-09,0,NaN,NaN,NaN,NaN,NaN,...,afternoon,NaN,0,0,0,23.7,1,no_rain,mild,NaN
2,speech therapy,13:20,F,2021-09-09,0,NaN,NaN,NaN,NaN,NaN,...,afternoon,NaN,0,0,0,23.7,1,no_rain,mild,NaN
3,physiotherapy,13:20,F,2021-09-09,0,NaN,NaN,NaN,NaN,NaN,...,afternoon,NaN,0,0,0,23.7,1,no_rain,mild,NaN
4,physiotherapy,14:00,M,2021-09-09,0,NaN,motor,2020-02-05,B. CAMBORIU,I67,...,afternoon,68,0,1,1,23.7,1,no_rain,mild,Other cerebrovascular diseases
